# Gemma 4 2B Fine-tuning — QLoRA (HuggingFace PEFT)

Fine-tunes `google/gemma-4-2b-it` on scene-reasoning examples using 4-bit QLoRA.
Standard `transformers` + `peft` + `trl` stack — no extra libraries.

**Requires T4 GPU** (Runtime → Change runtime type → T4 GPU)

## Workflow

1. **Cell 0** — Extract training data from your pipeline `reasoning.json` outputs → `data/finetune/train.jsonl`
2. **Cells 1–9** — Mount Drive, install deps, load model, train, save adapter

Training data format (one JSON object per line):
```json
{"messages": [{"role": "user", "content": "<scene prompt>"}, {"role": "assistant", "content": "<reasoning summary>"}]}
```

`prompt_used` and `summary` are already saved in every `reasoning.json` by the pipeline — no pipeline changes needed.

In [ ]:
# -- Cell 0: Extract training data from pipeline reasoning.json outputs
#
# Every reasoning.json already contains:
#   prompt_used  -> the scene prompt sent to Gemma  (user message)
#   summary      -> the model's response            (assistant message)
#
# This cell scans all outputs/ subdirectories, collects those pairs,
# filters low-quality entries, and writes data/finetune/train.jsonl.

import json
from pathlib import Path

OUTPUTS_DIR      = Path('outputs')          # where StreamingPipeline writes runs
OUT_JSONL        = Path('data/finetune/train.jsonl')
MIN_SUMMARY_LEN  = 20   # drop very short / empty summaries
MIN_PROMPT_LEN   = 30   # drop degenerate prompts

OUT_JSONL.parent.mkdir(parents=True, exist_ok=True)

examples = []
skipped  = 0

for reasoning_file in sorted(OUTPUTS_DIR.rglob('reasoning.json')):
    with open(reasoning_file) as f:
        data = json.load(f)

    for entry in data.get('outputs', []):
        prompt  = entry.get('prompt_used', '').strip()
        summary = entry.get('summary', '').strip()

        if len(prompt) < MIN_PROMPT_LEN or len(summary) < MIN_SUMMARY_LEN:
            skipped += 1
            continue

        examples.append({
            'messages': [
                {'role': 'user',      'content': prompt},
                {'role': 'assistant', 'content': summary},
            ]
        })

with open(OUT_JSONL, 'w') as f:
    for ex in examples:
        f.write(json.dumps(ex) + '\n')

print(f'Collected : {len(examples)} examples')
print(f'Skipped   : {skipped} (too short)')
print(f'Written to: {OUT_JSONL}')

if examples:
    print('\n--- First example ---')
    for msg in examples[0]['messages']:
        label = msg['role'].upper()
        text  = msg['content'][:300] + ('...' if len(msg['content']) > 300 else '')
        print(f'[{label}]\n{text}\n')

In [ ]:
# -- Cell 1: Mount Google Drive and navigate to project
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT_DIR = '/content/drive/MyDrive/Fusion_sensor'  # edit if needed
os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# -- Cell 2: Install PEFT / TRL extras
# torch + CUDA are already installed from 00_setup; add PEFT/TRL here
!pip install 'peft>=0.14.0' 'trl>=0.12.0' 'datasets>=3.0.0' -q
print('Done.')

In [ ]:
# -- Cell 3: Verify GPU
import torch

assert torch.cuda.is_available(), 'No GPU -- switch runtime to T4'
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU:  {torch.cuda.get_device_name(0)}')
print(f'VRAM: {vram_gb:.1f} GB')
assert vram_gb >= 10.0, f'Need >= 10 GB VRAM for QLoRA, got {vram_gb:.1f} GB'
print('GPU check passed')

In [ ]:
# -- Cell 4: Configuration
from pathlib import Path

MODEL_ID          = 'google/gemma-4-2b-it'

# Path to your JSONL training file (relative to PROJECT_DIR)
TRAIN_DATA_PATH   = 'data/finetune/train.jsonl'  # edit

# Where to save the LoRA adapter — on Drive so it persists across sessions
ADAPTER_SAVE_PATH = '/content/drive/MyDrive/Fusion_sensor/ckpt/gemma4_lora'

# LoRA
LORA_R       = 16
LORA_ALPHA   = 32    # convention: 2 * r
LORA_DROPOUT = 0.05

# Training
MAX_SEQ_LEN   = 1024  # scene prompts are short; raise if truncation warnings appear
BATCH_SIZE    = 1     # T4 15 GB -- keep at 1
GRAD_ACCUM    = 8     # effective batch size = 8
LEARNING_RATE = 2e-4
NUM_EPOCHS    = 3
WARMUP_RATIO  = 0.03

print('Config ready.')

In [ ]:
# -- Cell 5: Load and inspect training data
from datasets import load_dataset

dataset = load_dataset('json', data_files=TRAIN_DATA_PATH, split='train')
print(f'Training examples: {len(dataset)}')
print('\nFirst example:')
for msg in dataset[0]['messages']:
    role = msg['role'].upper()
    text = msg['content'][:200] + ('...' if len(msg['content']) > 200 else '')
    print(f'  [{role}] {text}')

# Sanity: every example must have exactly 2 messages (user, assistant)
bad = [i for i, ex in enumerate(dataset) if len(ex['messages']) != 2]
assert not bad, f'Examples with wrong message count: {bad[:5]}'
print('\nDataset OK.')

In [ ]:
# -- Cell 6: Load tokenizer + model in 4-bit (QLoRA)
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',          # NF4 is best for QLoRA
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,     # saves ~0.4 GB extra
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'right'        # required for causal LM training

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=torch.bfloat16,
)
model.config.use_cache = False          # must disable when using gradient checkpointing

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f'Trainable params before LoRA: {trainable:,} / {total:,}')

In [ ]:
# -- Cell 7: Apply LoRA adapter
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    task_type='CAUSAL_LM',
    # Gemma 4 attention + MLP projection layers
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj',
                    'gate_proj', 'up_proj', 'down_proj'],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
# -- Cell 8: Train with SFTTrainer
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='/content/checkpoints',
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type='cosine',
    bf16=True,
    logging_steps=5,
    save_strategy='epoch',
    optim='paged_adamw_8bit',           # 8-bit Adam saves ~2 GB VRAM
    max_seq_length=MAX_SEQ_LEN,
    packing=False,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    args=training_args,
)

trainer.train()
print('Training complete.')

In [ ]:
# -- Cell 9: Save LoRA adapter to Drive
Path(ADAPTER_SAVE_PATH).mkdir(parents=True, exist_ok=True)
model.save_pretrained(ADAPTER_SAVE_PATH)
tokenizer.save_pretrained(ADAPTER_SAVE_PATH)
print(f'Adapter saved to {ADAPTER_SAVE_PATH}')

In [ ]:
# -- Cell 10: Quick inference test with the fine-tuned adapter
model.config.use_cache = True  # re-enable for inference
model.eval()

TEST_PROMPT = (
    'Scene: 2 active tracks. Track 1 (car) at depth 12.3 m, moving forward. '
    'Track 2 (person) at depth 5.1 m, stationary. High BEV occupancy ahead. '
    'Describe the scene and recommend an action.'
)

messages = [{'role': 'user', 'content': TEST_PROMPT}]
input_ids = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    return_tensors='pt',
).to(model.device)

with torch.inference_mode():
    output_ids = model.generate(
        input_ids,
        max_new_tokens=256,
        do_sample=False,
    )

response = tokenizer.decode(
    output_ids[0][input_ids.shape[1]:],
    skip_special_tokens=True,
)
print('=== Model response ===')
print(response)

## Loading the adapter in the pipeline

To use the fine-tuned adapter in `GemmaReasoningWrapper`:

```python
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(model_id, ...)
model = PeftModel.from_pretrained(base_model, ADAPTER_SAVE_PATH)
```

Add a `lora_adapter_path` parameter to `GemmaReasoningWrapper.__init__` and load the adapter there.